<a href="https://colab.research.google.com/github/shahbazmaryam25/Data-Science-and-Generative-AI/blob/main/week_6_6_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
# upload an in-built Python (OK semi-in-built) dataset
from sklearn.datasets import load_diabetes

import pandas as pd
import numpy as np

# import the data
data = load_diabetes()
data

{'data': array([[ 0.03807591,  0.05068012,  0.06169621, ..., -0.00259226,
          0.01990749, -0.01764613],
        [-0.00188202, -0.04464164, -0.05147406, ..., -0.03949338,
         -0.06833155, -0.09220405],
        [ 0.08529891,  0.05068012,  0.04445121, ..., -0.00259226,
          0.00286131, -0.02593034],
        ...,
        [ 0.04170844,  0.05068012, -0.01590626, ..., -0.01107952,
         -0.04688253,  0.01549073],
        [-0.04547248, -0.04464164,  0.03906215, ...,  0.02655962,
          0.04452873, -0.02593034],
        [-0.04547248, -0.04464164, -0.0730303 , ..., -0.03949338,
         -0.00422151,  0.00306441]]),
 'target': array([151.,  75., 141., 206., 135.,  97., 138.,  63., 110., 310., 101.,
         69., 179., 185., 118., 171., 166., 144.,  97., 168.,  68.,  49.,
         68., 245., 184., 202., 137.,  85., 131., 283., 129.,  59., 341.,
         87.,  65., 102., 265., 276., 252.,  90., 100.,  55.,  61.,  92.,
        259.,  53., 190., 142.,  75., 142., 155., 225.,  59

In [28]:
# Convert data to PyTorch tensors
X = torch.tensor(data.data, dtype=torch.float32)
y = torch.tensor(data.target, dtype=torch.float32).reshape(-1, 1) # Reshape y to be a column vector


In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch.Size([353, 10]) torch.Size([353, 1])
torch.Size([89, 10]) torch.Size([89, 1])


In [30]:
from torch.utils.data import TensorDataset, DataLoader

# Create TensorDatasets and DataLoaders
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)

In [31]:
import torch
import torch.nn as nn

# Define the model
class DiabetesModel(nn.Module):
    def __init__(self):
        super(DiabetesModel, self).__init__()
        # we'll set up the layers as a sequence using nn.Sequential
        self.layers = nn.Sequential(

            # first layer will be a linear layer that has 5x neurons
            # (5x sets of linear regression)
            # the layer takes the 10 features as input (i.e. 10, 5)
            nn.Linear(10, 5),

            nn.ReLU(), # ReLU activation

            # second linear layer again has 5 neurons
            # this time taking the input as the output of the last layer
            # (which had 5x neurons)
            nn.Linear(5, 5),

            nn.ReLU(), # ReLU again

            # last linear layer takes the output from the previous 5 neurons
            # this time its a single output with no activation
            # i.e. this is the predicitons (regression)
            nn.Linear(5, 1)
        )

    def forward(self, x):
        return self.layers(x) # pass the data through the layers

In [34]:
import sys
from unittest.mock import MagicMock

# Mock torch._dynamo to bypass the internal PyTorch bug in Python 3.13
sys.modules['torch._dynamo'] = MagicMock()

import torch.optim as optim

# Initialize the model, loss function, and optimizer
model = DiabetesModel()
criterion = nn.MSELoss() # MSE loss function
optimiser = optim.Adam(model.parameters(), lr=0.001)

In [35]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training loop (example - you'll likely want to add more epochs)
epochs = 100 # 100 epochs

for epoch in range(epochs):
  # use the train_loader to pass the inputs (x) and targets (y)
  for inputs, targets in train_loader:
    # pass to the GPU (hopefully)
    inputs, targets = inputs.to(device), targets.to(device)

    # pass model to GPU as well
    model.to(device)

    model.train() # put the model object in train mode
    optimiser.zero_grad() # reset the gradiants
    outputs = model(inputs) # create outputs
    loss = criterion(outputs, targets) # compare with Y to get loss
    loss.backward() # backpropogate the loss (next week)
    optimiser.step() # # update the parameters based on this round of training

  # every 10 steps we will print out the current loss
    if (epoch+1) % 10 == 0: # modular arithmetic
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

Epoch [10/100], Loss: 34243.4688
Epoch [10/100], Loss: 28352.0254
Epoch [10/100], Loss: 33280.8516
Epoch [10/100], Loss: 32466.5117
Epoch [10/100], Loss: 30042.9746
Epoch [10/100], Loss: 24468.5332
Epoch [10/100], Loss: 25235.0195
Epoch [10/100], Loss: 42023.0039
Epoch [20/100], Loss: 31971.2695
Epoch [20/100], Loss: 34567.2305
Epoch [20/100], Loss: 28190.5723
Epoch [20/100], Loss: 25631.9668
Epoch [20/100], Loss: 25824.6895
Epoch [20/100], Loss: 30785.1855
Epoch [20/100], Loss: 33410.3398
Epoch [20/100], Loss: 3825.3184
Epoch [30/100], Loss: 27670.3379
Epoch [30/100], Loss: 32671.4375
Epoch [30/100], Loss: 27342.3242
Epoch [30/100], Loss: 28527.9746
Epoch [30/100], Loss: 26202.2734
Epoch [30/100], Loss: 30329.5176
Epoch [30/100], Loss: 36114.7656
Epoch [30/100], Loss: 29202.3125
Epoch [40/100], Loss: 35106.25
Epoch [40/100], Loss: 28910.3633
Epoch [40/100], Loss: 28522.5801
Epoch [40/100], Loss: 31526.4492
Epoch [40/100], Loss: 22215.1191
Epoch [40/100], Loss: 27500.3066
Epoch [40/100

In [36]:
# Evaluation (example)
model.eval() # testing mode
mse_values = [] # collect the MSE scores

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs) # predict the test data

        # Calculate Mean Squared Error
        mse = criterion(outputs, targets) # calcualte mse for the batch
        mse_values.append(mse.item()) # add to the list of MSE values

# Calculate and print the average MSE
avg_mse = np.mean(mse_values)
print(f"Average MSE on test set: {avg_mse}")

Average MSE on test set: 26297.4169921875


In [37]:
# Evaluation
model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predictions.extend(outputs.cpu().numpy())
        actuals.extend(targets.cpu().numpy())

# Create DataFrame
results_df = pd.DataFrame({'Predicted': np.array(predictions).flatten(), 'Actual': np.array(actuals).flatten()})
results_df


,Predicted,Actual
0,-0.379338,219.0
1,-0.395951,70.0
2,-0.379340,202.0
3,-0.378548,230.0
4,-0.387304,111.0
...,...,...
84,-0.395414,153.0
85,-0.395029,98.0
86,-0.395392,37.0
87,-0.394962,63.0


In [38]:
import torch.nn as nn


class RegressionModel(nn.Module):

  def __init__(self, input_dim):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(input_dim, 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 1),  # Output layer for regression
    )

  def forward(self, x):
    return self.net(x)

Increasing the number of epochs to 1,000 to check if it gives better results:

In [41]:
epochs = 1000

for epoch in range(epochs):
  model.train()  # Ensure model is in train mode
  epoch_loss = 0

  # We must iterate through our training data batches
  for inputs, targets in train_loader:
    inputs, targets = inputs.to(device), targets.to(device)

    # 1. Forward pass
    outputs = model(inputs)
    loss = criterion(outputs, targets)

    # 2. Backward pass and optimization using the defined 'optimiser'
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()

    epoch_loss += loss.item() * inputs.size(0)

  # Print progress every 100 epochs to check stability
  if (epoch + 1) % 100 == 0:
    avg_loss = epoch_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

Epoch [100/1000], Loss: 29831.5537
Epoch [200/1000], Loss: 29831.5535
Epoch [300/1000], Loss: 29831.5549
Epoch [400/1000], Loss: 29831.5539
Epoch [500/1000], Loss: 29831.5538
Epoch [600/1000], Loss: 29831.5540
Epoch [700/1000], Loss: 29831.5539
Epoch [800/1000], Loss: 29831.5533
Epoch [900/1000], Loss: 29831.5539
Epoch [1000/1000], Loss: 29831.5541


Experimenting with the architecture (number of neurons and/or number of layers). Checking to see if reaching an optimal architecture is possible:

In [42]:
import torch
import torch.nn as nn


class FlexibleRegressionModel(nn.Module):

  def __init__(self, input_dim, hidden_dims=[64, 32]):
    super().__init__()

    layers = []
    prev_dim = input_dim

    # Dynamically build hidden layers based on the list
    for h_dim in hidden_dims:
      layers.append(nn.Linear(prev_dim, h_dim))
      layers.append(nn.ReLU())
      prev_dim = h_dim

    # Final output layer for regression
    layers.append(nn.Linear(prev_dim, 1))

    self.network = nn.Sequential(*layers)

  def forward(self, x):
    return self.network(x)


# Example usage: Try a deeper or wider architecture easily
# model = FlexibleRegressionModel(input_dim=X.shape[1], hidden_dims=[128, 64, 32])